# Fake News Detector

## Installing Necessary Libraries

In [48]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import string
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

# new
from googletrans import Translator
from langdetect import detect

## Loading the data

In [49]:
data_fake=pd.read_csv('Fake.csv')
data_true=pd.read_csv('True.csv')

### Data Preview 

In [50]:
data_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [51]:
data_fake.tail()

,title,text,subject,date
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016"
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016"
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016"
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016"
23480,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016"


In [52]:
data_true.tail()

,title,text,subject,date
21412,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017"
21413,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017"
21414,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017"
21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017"
21416,Indonesia to buy $1.14 billion worth of Russia...,JAKARTA (Reuters) - Indonesia will buy 11 Sukh...,worldnews,"August 22, 2017"


In [53]:
data_fake["class"]=0
data_true['class']=1

In [54]:
data_fake.shape, data_true.shape

((23481, 5), (21417, 5))

In [55]:
data_fake_manual_testing = data_fake.tail(10)
for i in range(23480,23470,-1):
    data_fake.drop([i],axis = 0, inplace = True)

    
data_true_manual_testing = data_true.tail(10)
for i in range(21416,21406,-1):
    data_true.drop([i],axis = 0, inplace = True)
    
    

In [56]:
data_fake.shape, data_true.shape

((23471, 5), (21407, 5))

In [57]:
# Correct way
data_fake_manual_testing.loc[:, 'class'] = 0
data_true_manual_testing.loc[:, 'class'] = 1

In [58]:
data_fake_manual_testing.head(10)

,title,text,subject,date,class
23471,Seven Iranians freed in the prisoner swap have...,"21st Century Wire says This week, the historic...",Middle-east,"January 20, 2016",0
23472,#Hashtag Hell & The Fake Left,By Dady Chery and Gilbert MercierAll writers ...,Middle-east,"January 19, 2016",0
23473,Astroturfing: Journalist Reveals Brainwashing ...,Vic Bishop Waking TimesOur reality is carefull...,Middle-east,"January 19, 2016",0
23474,The New American Century: An Era of Fraud,Paul Craig RobertsIn the last years of the 20t...,Middle-east,"January 19, 2016",0
23475,Hillary Clinton: ‘Israel First’ (and no peace ...,Robert Fantina CounterpunchAlthough the United...,Middle-east,"January 18, 2016",0
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",0
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",0
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",0
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",0
23480,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",0


In [59]:
data_true_manual_testing.head(10)

,title,text,subject,date,class
21407,"Mata Pires, owner of embattled Brazil builder ...","SAO PAULO (Reuters) - Cesar Mata Pires, the ow...",worldnews,"August 22, 2017",1
21408,"U.S., North Korea clash at U.N. forum over nuc...",GENEVA (Reuters) - North Korea and the United ...,worldnews,"August 22, 2017",1
21409,"U.S., North Korea clash at U.N. arms forum on ...",GENEVA (Reuters) - North Korea and the United ...,worldnews,"August 22, 2017",1
21410,Headless torso could belong to submarine journ...,COPENHAGEN (Reuters) - Danish police said on T...,worldnews,"August 22, 2017",1
21411,North Korea shipments to Syria chemical arms a...,UNITED NATIONS (Reuters) - Two North Korean sh...,worldnews,"August 21, 2017",1
21412,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017",1
21413,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017",1
21414,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017",1
21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017",1
21416,Indonesia to buy $1.14 billion worth of Russia...,JAKARTA (Reuters) - Indonesia will buy 11 Sukh...,worldnews,"August 22, 2017",1


In [60]:
data_merge=pd.concat([data_fake, data_true], axis = 0)
data_merge.head(10)

,title,text,subject,date,class
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0
5,Racist Alabama Cops Brutalize Black Boy While...,The number of cases of cops brutalizing and ki...,News,"December 25, 2017",0
6,"Fresh Off The Golf Course, Trump Lashes Out A...",Donald Trump spent a good portion of his day a...,News,"December 23, 2017",0
7,Trump Said Some INSANELY Racist Stuff Inside ...,In the wake of yet another court decision that...,News,"December 23, 2017",0
8,Former CIA Director Slams Trump Over UN Bully...,Many people have raised the alarm regarding th...,News,"December 22, 2017",0
9,WATCH: Brand-New Pro-Trump Ad Features So Muc...,Just when you might have thought we d get a br...,News,"December 21, 2017",0


#### "title",  "subject" and "date" columns is not required for detecting the fake news, so I am going to drop the columns.

In [61]:
data_merge.columns

Index(['title', 'text', 'subject', 'date', 'class'], dtype='object')

In [62]:
data=data_merge.drop(['title','subject','date'], axis = 1)

In [63]:
#count of missing values
data.isnull().sum() 

text     0
class    0
dtype: int64

#### Randomly shuffling the dataframe 

In [64]:
data = data.sample(frac = 1)

In [65]:
data.head()

,text,class
8510,"Jon Ritzheimer, infamous hater of Muslims and ...",0
11906,"(Reuters) - A trio of U.S., Japanese and Russi...",1
7543,WASHINGTON (Reuters) - Several intriguing scen...,1
9713,ISLAMABAD (Reuters) - Pakistan angrily critici...,1
89,"On Thursday, the Pentagon s official Twitter a...",0


In [66]:
data.reset_index(inplace = True)
data.drop(['index'], axis = 1, inplace = True)

In [67]:
data.columns

Index(['text', 'class'], dtype='object')

In [68]:
data.head()

,text,class
0,"Jon Ritzheimer, infamous hater of Muslims and ...",0
1,"(Reuters) - A trio of U.S., Japanese and Russi...",1
2,WASHINGTON (Reuters) - Several intriguing scen...,1
3,ISLAMABAD (Reuters) - Pakistan angrily critici...,1
4,"On Thursday, the Pentagon s official Twitter a...",0


## Preprocessing Text

#### Creating a function to convert the text in lowercase, remove the extra space, special chr., ulr and links.

In [69]:
def wordopt(text):
    text = text.lower()
    text = re.sub('\[.*?\]','',text)
    text = re.sub("\\W"," ",text)
    text = re.sub('https?://\S+|www\.\S+','',text)
    text = re.sub('<.*?>+',b'',text)
    text = re.sub('[%s]' % re.escape(string.punctuation),'',text)
    text = re.sub('\w*\d\w*','',text)
    return text

In [70]:
data['text'] = data['text'].apply(wordopt)

#### Defining dependent and independent variable as x and y

In [71]:
x = data['text']
y = data['class']

## Training the model

#### Splitting the dataset into training set and testing set. 

In [72]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.25)

### Extracting Features from the Text

#### Convert text to vectors

In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)

## Logistic Regression

In [74]:
from sklearn.linear_model import LogisticRegression

In [75]:
LR = LogisticRegression()
LR.fit(xv_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [76]:
pred_lr = LR.predict(xv_test)

In [77]:
LR.score(xv_test, y_test)

0.9864527629233512

In [78]:
print (classification_report(y_test, pred_lr))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5871
           1       0.98      0.99      0.99      5349

    accuracy                           0.99     11220
   macro avg       0.99      0.99      0.99     11220
weighted avg       0.99      0.99      0.99     11220



## Decision Tree Classifier

In [79]:
from sklearn.tree import DecisionTreeClassifier

DT = DecisionTreeClassifier()
DT.fit(xv_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [80]:
pred_dt = DT.predict(xv_test)

In [81]:
DT.score(xv_test, y_test)

0.9947415329768271

In [82]:
print (classification_report(y_test, pred_lr))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5871
           1       0.98      0.99      0.99      5349

    accuracy                           0.99     11220
   macro avg       0.99      0.99      0.99     11220
weighted avg       0.99      0.99      0.99     11220



## Gradient Boost Classifier

In [83]:
from sklearn.ensemble import GradientBoostingClassifier

GB = GradientBoostingClassifier(random_state = 0)
GB.fit(xv_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [84]:
pred_gb = GB.predict(xv_test)

In [85]:
GB.score(xv_test, y_test)

0.9951871657754011

In [86]:
print(classification_report(y_test, pred_gb))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00      5871
           1       0.99      1.00      0.99      5349

    accuracy                           1.00     11220
   macro avg       1.00      1.00      1.00     11220
weighted avg       1.00      1.00      1.00     11220



## Random Forest Classifier

In [87]:
from sklearn.ensemble import RandomForestClassifier

RF = RandomForestClassifier(random_state = 0)
RF.fit(xv_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [88]:
pred_rf = RF.predict(xv_test)

In [89]:
RF.score(xv_test, y_test)

0.9866310160427807

In [90]:
print (classification_report(y_test, pred_rf))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5871
           1       0.99      0.99      0.99      5349

    accuracy                           0.99     11220
   macro avg       0.99      0.99      0.99     11220
weighted avg       0.99      0.99      0.99     11220



In [91]:
from sklearn.svm import SVC
SVM = SVC(kernel='linear', random_state=0)  # linear kernel works well for text
SVM.fit(xv_train, y_train)

pred_svm = SVM.predict(xv_test)

In [92]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def evaluate_model(model, xv_test, y_test, model_name):
    pred = model.predict(xv_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    print(f"{model_name} Results:")
    print(f"Accuracy: {acc*100:.2f}%")
    print(f"F1 Score: {f1*100:.2f}%")
    print(classification_report(y_test, pred))
    print("-"*50)
    return acc, f1

# Evaluate all models
evaluate_model(LR, xv_test, y_test, "Logistic Regression")
evaluate_model(DT, xv_test, y_test, "Decision Tree")
evaluate_model(GB, xv_test, y_test, "Gradient Boosting")
evaluate_model(RF, xv_test, y_test, "Random Forest")
evaluate_model(SVM, xv_test, y_test, "Support Vector Machine")

Logistic Regression Results:
Accuracy: 98.65%
F1 Score: 98.58%
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5871
           1       0.98      0.99      0.99      5349

    accuracy                           0.99     11220
   macro avg       0.99      0.99      0.99     11220
weighted avg       0.99      0.99      0.99     11220

--------------------------------------------------
Decision Tree Results:
Accuracy: 99.47%
F1 Score: 99.45%
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      5871
           1       0.99      0.99      0.99      5349

    accuracy                           0.99     11220
   macro avg       0.99      0.99      0.99     11220
weighted avg       0.99      0.99      0.99     11220

--------------------------------------------------
Gradient Boosting Results:
Accuracy: 99.52%
F1 Score: 99.50%
              precision    recall  f1-score   support

        

(0.99349376114082, 0.9931833037631899)

## Testing the Model

In [93]:
def output_lable(n):
    if n==0:
        return "Fake News"
    elif n==1:
        return "Not A Fake News"

     
translator = Translator()

def manual_testing(news):
    # Detect language
    lang = detect(news)
    
    # Translate to English if the input is Hindi
    if lang == 'hi':
        news_translated = translator.translate(news, src='hi', dest='en').text
    else:
        news_translated = news
    
    # Prepare dataframe for vectorization
    testing_news = {"text": [news_translated]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test['text'] = new_def_test["text"].apply(wordopt)
    
    # Transform input text using TF-IDF
    new_x_test = new_def_test["text"]
    new_xv_test = vectorization.transform(new_x_test)
    
    # Make predictions
    pred_LR = LR.predict(new_xv_test)
    pred_DT = DT.predict(new_xv_test)
    pred_GB = GB.predict(new_xv_test)
    pred_RF = RF.predict(new_xv_test)
    pred_SVM = SVM.predict(new_xv_test)
    
    # Print results
    print("\n\nLR Prediction: {} \nDT Prediction: {} \nGBC Prediction: {} \nRFC Prediction: {} \nSVM Prediction: {}".format(
        output_lable(pred_LR[0]),
        output_lable(pred_DT[0]),
        output_lable(pred_GB[0]),
        output_lable(pred_RF[0]),
        output_lable(pred_SVM[0])
    ))


### Model Testing With Manual Entry

In [94]:
news = str(input()) 
manual_testing(news)

 भाजपा ने धर्मेंद्र प्रधान को बिहार का चुनाव प्रभारी बनाया:भूपेंद्र यादव को बंगाल और बैजयंत पांडा को तमिलनाडु की जिम्मेदारी भाजपा ने गुरुवार को बिहार, पश्चिम बंगाल और तमिलनाडु में विधानसभा चुनाव के लिए प्रभारी और सह प्रभारी नियुक्त किए हैं। बिहार के लिए केंद्रीय शिक्षा मंत्री धर्मेंद्र प्रधान को चुनाव प्रभारी, उत्तर प्रदेश के उप मुख्यमंत्री केशव प्रसाद मौर्य और केंद्रीय जल शक्ति मंत्री सीआर पाटिल को सह-प्रभारी बनाया है।  वहीं, केंद्रीय पर्यावरण, वन एवं जलवायु परिवर्तन मंत्री भूपेंद्र यादव को पश्चिम बंगाल विधानसभा चुनाव का प्रभारी और त्रिपुरा के पूर्व मुख्यमंत्री बिप्लब कुमार देब को सह प्रभारी बनाया गया है। इसके अलावा तमिलनाडु में भाजपा सांसद बैजयंत पांडा को प्रभारी और मुरलीधर मोहोल को सह प्रभारी नियुक्त किया गया है।  बिहार में अक्टूबर-नवंबर में विधानसभा चुनाव होने हैं, यहां फिलहाल नीतीश कुमार की JDU और भाजपा गठबंधन की सरकार है। वहीं, 2026 में मार्च से मई के बीच पश्चिम बंगाल और तमिलनाडु में चुनाव होंगे। बंगाल में ममता बनर्जी की पार्टी TMC और तमिलनाडु में द्रविड़ मुनेत्र कड़गम (DMK) सत्त



LR Prediction: Not A Fake News 
DT Prediction: Fake News 
GBC Prediction: Fake News 
RFC Prediction: Fake News 
SVM Prediction: Not A Fake News




LR Prediction: Not A Fake News 
DT Prediction: Not A Fake News 
GBC Prediction: Not A Fake News 
RFC Prediction: Not A Fake News


In [104]:
news=str(input())
manual_testing(news)

 सरकार ने किसानों के लिए नई योजना की घोषणा की।




LR Prediction: Fake News 
DT Prediction: Fake News 
GBC Prediction: Fake News 
RFC Prediction: Fake News 
SVM Prediction: Fake News




LR Prediction: Not A Fake News 
DT Prediction: Fake News 
GBC Prediction: Fake News 
RFC Prediction: Not A Fake News
